# CODY-SAM3 — Feature analysis for clinical interpretation

This notebook produces the per-phenomenology analyses that support the **Methods**
and **Results** sections of the paper. For each of the eight target
phenomenologies, it shows:

- which SAM 3 signals are most discriminative,
- how their distributions differ between positive and negative windows,
- where on the body the discriminative signal concentrates (anatomical mapping),
- effect sizes and statistical significance (Mann-Whitney U + Cohen's d),
- and the permutation importance of each feature in the trained TabICLv2 model.

The notebook produces all figures and tables in
`<RUNS_ROOT>/feature_analysis/`, ready to drop into the manuscript or
Supplementary Material.

**Inputs needed:**

- `MERGED_ROOT` — output of `merge_sam3_labels.py`.
- A trained tier-N run (`RUNS_ROOT/sam3_tier{tier}/`), with its cached
  feature table `features_windows_train__t<tier>__...csv.gz` and its trained
  per-label models.

Pick the tier you want to analyse below (typically Tier 2, the recommended one).

## 1. Setup

In [ ]:
from pathlib import Path
import sys, json, gzip, re, math
from typing import Dict, List, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Paths (edit to match)
RUNS_ROOT  = Path(r'C:\Users\<user>\Desktop\sam_3\runs')
TIER       = 2
BUNDLE_DIR = RUNS_ROOT / f'sam3_tier{TIER}'
OUT_DIR    = BUNDLE_DIR / 'feature_analysis'
OUT_DIR.mkdir(parents=True, exist_ok=True)

PIPELINE_DIR = BUNDLE_DIR.parent.parent  # heuristic; edit if needed
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

import sam3_features as s3f

print(f'Tier:         {TIER}')
print(f'Bundle dir:   {BUNDLE_DIR}')
print(f'Output dir:   {OUT_DIR}')

## 2. Load cached training window table

Each `train_sam3.py` run saves a feature-and-labels table
(`features_windows_train__t<tier>__...csv.gz`) with one row per training
window. We load it here for analysis.

In [ ]:
cache_glob = list(BUNDLE_DIR.glob(f'features_windows_train__t{TIER}__*.csv.gz'))
assert cache_glob, f'No training cache under {BUNDLE_DIR}. Run train_sam3.py first.'
cache_path = cache_glob[0]
print(f'Loading {cache_path.name} ...')
with gzip.open(cache_path, 'rt', encoding='utf-8') as f:
    dfw = pd.read_csv(f, low_memory=False,
                      dtype={'patient_id': 'string',
                             'From': 'Float64', 'To': 'Float64'})
print(f'Shape: {dfw.shape}')
print(f'Patients: {dfw.patient_id.nunique()}')
feat_cols = [c for c in dfw.columns if c.startswith('f__')]
ctx_cols  = [c for c in dfw.columns if c.startswith('ctx__')]
print(f'Features: {len(feat_cols)}  Context flags: {len(ctx_cols)}')

## 3. Feature catalogue with clinical descriptions

Every signal used by the tier is documented with a clinical-interpretation
sentence in `sam3_features.SIGNAL_DESCRIPTIONS`. We dump them to a CSV.

In [ ]:
# Each feature is f__<signal>__<descriptor>. Group by signal to list signals used.
sig_set = set()
for c in feat_cols:
    m = re.match(r'^f__(.*?)__(\w+)$', c)
    if m:
        sig_set.add(m.group(1))
signals = sorted(sig_set)
print(f'{len(signals)} signals used at Tier {TIER}')

catalogue = []
for sig in signals:
    catalogue.append(dict(
        signal      = sig,
        description = s3f.get_signal_description(sig),
    ))
cat_df = pd.DataFrame(catalogue)
cat_path = OUT_DIR / 'feature_catalogue.csv'
cat_df.to_csv(cat_path, index=False)
print(f'Wrote {cat_path}')
cat_df.head(10)

## 4. Univariate discrimination per phenomenology

For each of the eight phenomenologies and each feature `f__<signal>__<descriptor>`,
we compare positive windows (label=1) against negative windows (label=0)
with the Mann-Whitney U test and Cohen's d effect size. Higher |d|
indicates a feature that better separates positive from negative.

We aggregate at the signal level to surface which body locations / kinematic
channels are most informative.

In [ ]:
from numpy import nanmean, nanstd

DEFAULT_SYMPTOMS = ['Dystonia', 'Tremor', 'Myoclonus', 'Chorea',
                    'Athetosis', 'Ballismus', 'Stereotypies', 'Tics']

def cohens_d(x, y):
    """Cohen's d for two independent samples, NaN-safe."""
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]; y = y[np.isfinite(y)]
    if x.size < 2 or y.size < 2: return np.nan
    sd = math.sqrt(((x.size - 1) * nanstd(x) ** 2 +
                    (y.size - 1) * nanstd(y) ** 2) / (x.size + y.size - 2))
    if not (sd > 0): return np.nan
    return (nanmean(x) - nanmean(y)) / sd

def univariate_discrimination(dfw, feat_cols, symptoms):
    rows = []
    for sym in symptoms:
        if sym not in dfw.columns: continue
        y = pd.to_numeric(dfw[sym], errors='coerce').to_numpy(float)
        mask_pos = (y == 1)
        mask_neg = (y == 0)
        n_pos = int(mask_pos.sum()); n_neg = int(mask_neg.sum())
        if n_pos < 5 or n_neg < 5:
            print(f'  {sym}: too few labels (pos={n_pos}, neg={n_neg}), skipping')
            continue
        for fc in feat_cols:
            x = pd.to_numeric(dfw[fc], errors='coerce').to_numpy(float)
            xp = x[mask_pos]; xn = x[mask_neg]
            xp = xp[np.isfinite(xp)]; xn = xn[np.isfinite(xn)]
            if xp.size < 5 or xn.size < 5: continue
            try:
                u, p = stats.mannwhitneyu(xp, xn, alternative='two-sided')
            except Exception:
                u, p = np.nan, np.nan
            d = cohens_d(xp, xn)
            rows.append(dict(
                symptom=sym, feature=fc,
                n_pos=int(xp.size), n_neg=int(xn.size),
                mean_pos=float(nanmean(xp)), mean_neg=float(nanmean(xn)),
                cohens_d=float(d) if np.isfinite(d) else np.nan,
                abs_d=float(abs(d)) if np.isfinite(d) else np.nan,
                mw_p=float(p) if np.isfinite(p) else np.nan,
            ))
    return pd.DataFrame(rows)

uni = univariate_discrimination(dfw, feat_cols, DEFAULT_SYMPTOMS)
uni.to_csv(OUT_DIR / 'univariate_discrimination.csv', index=False)
print(f'Wrote {OUT_DIR}/univariate_discrimination.csv  rows={len(uni)}')

## 5. Top discriminative features per phenomenology

For each symptom, surface the top 15 most discriminative features (by
|Cohen's d|, Bonferroni-corrected). Each feature is also mapped back to its
signal description for clinical interpretability.

In [ ]:
TOP_N = 15
BONF_FACTOR = max(1, dfw.shape[0])  # rough conservative correction

def signal_of(feat_col: str) -> str:
    m = re.match(r'^f__(.*?)__(\w+)$', feat_col)
    return m.group(1) if m else feat_col
def descriptor_of(feat_col: str) -> str:
    m = re.match(r'^f__(.*?)__(\w+)$', feat_col)
    return m.group(2) if m else ''

top_rows = []
for sym in DEFAULT_SYMPTOMS:
    sub = uni[uni.symptom == sym].dropna(subset=['abs_d'])
    if sub.empty: continue
    sub = sub.sort_values('abs_d', ascending=False).head(TOP_N).copy()
    sub['signal'] = sub.feature.apply(signal_of)
    sub['descriptor'] = sub.feature.apply(descriptor_of)
    sub['signal_description'] = sub.signal.apply(s3f.get_signal_description)
    sub['bonferroni_signif'] = sub.mw_p < (0.05 / len(feat_cols))
    top_rows.append(sub)
    print(f'\n=== {sym} ===')
    print(sub[['feature','cohens_d','mw_p','signal_description']].to_string(index=False)[:1800])

if top_rows:
    top_df = pd.concat(top_rows, ignore_index=True)
    top_df.to_csv(OUT_DIR / 'top_features_per_symptom.csv', index=False)
    print(f'\nWrote {OUT_DIR}/top_features_per_symptom.csv')

## 6. Distribution plots — top features

Box+strip plots comparing the distribution of each top feature between
positive and negative windows. Sanity check that the discrimination is real
and not driven by outliers.

In [ ]:
import seaborn as sns
sns.set_style('whitegrid')

for sym in DEFAULT_SYMPTOMS:
    sub_top = top_df[top_df.symptom == sym].head(8)
    if sub_top.empty: continue
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(f'{sym} — top {len(sub_top)} discriminative features', fontsize=14)
    for ax, (_, r) in zip(axes.flatten(), sub_top.iterrows()):
        fc = r.feature
        y_raw = pd.to_numeric(dfw[sym], errors='coerce').to_numpy(float)
        x_raw = pd.to_numeric(dfw[fc], errors='coerce').to_numpy(float)
        ok = np.isfinite(x_raw) & np.isin(y_raw, [0, 1])
        if not ok.any():
            ax.set_visible(False); continue
        plot_df = pd.DataFrame({'value': x_raw[ok], 'group': y_raw[ok].astype(int)})
        sns.boxplot(data=plot_df, x='group', y='value', ax=ax,
                    palette={0: 'lightgray', 1: 'salmon'}, fliersize=0)
        ax.set_title(fc[:30] + ('...' if len(fc) > 30 else ''), fontsize=9)
        ax.set_xticklabels(['neg', 'pos'])
        ax.set_xlabel(''); ax.set_ylabel('')
    for ax in axes.flatten()[len(sub_top):]:
        ax.set_visible(False)
    plt.tight_layout()
    out = OUT_DIR / f'distributions_{sym}.png'
    fig.savefig(out, dpi=120, bbox_inches='tight')
    plt.close(fig)
    print(f'Wrote {out}')

## 7. Anatomical body map — where each phenomenology localizes

Aggregate the |Cohen's d| effect sizes by **anatomical region** (head,
shoulders, arms, trunk, legs). We overlay these on a schematic body figure
to visualise where on the body each phenomenology is most discriminative.

This is the figure that goes in the Methods section of the paper to show
the interpretability advantage of SAM 3 over keypoint-only methods.

In [ ]:
REGION_KEYWORDS = {
    'head':             ['head', 'nose'],
    'left_shoulder':    ['left_shoulder', 'lsh'],
    'right_shoulder':   ['right_shoulder', 'rsh'],
    'left_arm':         ['left_arm', 'larm'],
    'right_arm':        ['right_arm', 'rarm'],
    'trunk':            ['centroid', 'major_axis', 'minor_axis',
                         'orientation', 'trunk', 'torsion',
                         'silhouette', 'aspect', 'solidity', 'extent',
                         'area', 'perimeter', 'bbox', 'middle'],
    'pelvis':           ['hip'],
    'left_leg':         ['feet_left', 'left_leg', 'leg_left', 'bottom_left'],
    'right_leg':        ['feet_right', 'right_leg', 'leg_right', 'bottom_right'],
}

def feature_region(feat_col):
    s = signal_of(feat_col).lower()
    for region, kws in REGION_KEYWORDS.items():
        if any(kw in s for kw in kws):
            return region
    return 'other'

uni['region'] = uni.feature.apply(feature_region)
agg = uni.groupby(['symptom', 'region'])['abs_d'].mean().reset_index()
agg_pivot = agg.pivot(index='region', columns='symptom', values='abs_d').fillna(0)
agg_pivot.to_csv(OUT_DIR / 'region_discriminativity_heatmap.csv')

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(agg_pivot, annot=True, fmt='.2f', cmap='Reds', ax=ax)
ax.set_title('Mean |Cohen\'s d| per anatomical region and phenomenology')
plt.tight_layout()
fig.savefig(OUT_DIR / 'region_discriminativity_heatmap.png',
            dpi=120, bbox_inches='tight')
plt.close(fig)
print(f'Wrote heatmap')

## 8. Permutation importance from the trained TabICLv2 models

While Cohen's d gives a univariate view, permutation importance captures the
marginal contribution of each feature to the trained model's predictions.
Together they give a complete picture of feature usefulness.

Per phenomenology, we shuffle one feature at a time and measure the drop in
ROC-AUC on a held-out patient-grouped fold. Top 10 features are saved.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import joblib

RUN_PERM = False  # toggle: this is slow (5-30 min per label)

if RUN_PERM:
    feat_cols_with_ctx = feat_cols + ctx_cols
    perm_rows = []
    for sym in DEFAULT_SYMPTOMS:
        model_path = BUNDLE_DIR / 'models' / f'label_{sym}' / 'model.joblib'
        if not model_path.exists(): continue
        y = pd.to_numeric(dfw[sym], errors='coerce').to_numpy(float)
        known = np.isin(y, [0, 1])
        if known.sum() < 50: continue
        Xall = dfw.loc[known, feat_cols_with_ctx].fillna(0).to_numpy(float)
        yall = y[known].astype(int)
        pid = dfw.loc[known, 'patient_id'].astype(str).to_numpy()
        # Single held-out fold for speed
        gkf = GroupKFold(n_splits=5)
        for tr, te in gkf.split(Xall, yall, groups=pid):
            break
        try:
            from tabicl import TabICLClassifier
            clf = TabICLClassifier.load(model_path)
        except Exception:
            clf = joblib.load(model_path)
        # Baseline AUC
        base_auc = roc_auc_score(yall[te], clf.predict_proba(Xall[te])[:, 1])
        print(f'  {sym}: base AUC={base_auc:.3f}')
        # Permute each feature
        rng = np.random.default_rng(42)
        for j, fc in enumerate(feat_cols_with_ctx):
            X_shuf = Xall[te].copy()
            X_shuf[:, j] = rng.permutation(X_shuf[:, j])
            auc_s = roc_auc_score(yall[te], clf.predict_proba(X_shuf)[:, 1])
            perm_rows.append(dict(
                symptom=sym, feature=fc,
                base_auc=base_auc, shuffled_auc=auc_s,
                importance=base_auc - auc_s,
            ))
    if perm_rows:
        perm_df = pd.DataFrame(perm_rows)
        perm_df.to_csv(OUT_DIR / 'permutation_importance.csv', index=False)
        print(f'Wrote permutation_importance.csv ({len(perm_df)} rows)')
else:
    print('RUN_PERM=False — set to True to run permutation importance.')

## 9. Paper-ready summary table

Combine all per-symptom findings into a single multi-sheet Excel file that
is ready to drop into the manuscript or Supplementary Material.

In [ ]:
from openpyxl import Workbook
wb_path = OUT_DIR / 'paper_summary.xlsx'
with pd.ExcelWriter(wb_path, engine='openpyxl') as w:
    cat_df.to_excel(w, sheet_name='Feature catalogue', index=False)
    if not uni.empty:
        uni.to_excel(w, sheet_name='Univariate discrimination', index=False)
    if 'top_df' in dir() and not top_df.empty:
        top_df.to_excel(w, sheet_name='Top features per symptom', index=False)
    agg_pivot.to_excel(w, sheet_name='Region heatmap')
print(f'Wrote paper summary -> {wb_path}')

---

**End of feature analysis.** Outputs in `OUT_DIR`:

- `feature_catalogue.csv` — clinical description of each signal used.
- `univariate_discrimination.csv` — Mann-Whitney + Cohen's d per (feature, symptom).
- `top_features_per_symptom.csv` — top 15 features per phenomenology.
- `distributions_<symptom>.png` — boxplots of top features per symptom.
- `region_discriminativity_heatmap.{csv,png}` — anatomical-region heatmap.
- `permutation_importance.csv` (if `RUN_PERM=True`).
- `paper_summary.xlsx` — everything in one Excel workbook for the manuscript.